# Module 2: 核心数据结构

## 学习目标
- 理解 `SamplingParams` 采样参数
- 掌握 `Req` (Request) 类的设计
- 理解 `Batch` 批处理类
- 学习 `Context` 全局上下文

---

## 2.1 代码位置

这些核心数据结构定义在 `python/minisgl/core.py` 中，是整个系统的基础。

```
mini-sglang/
└── python/minisgl/
    └── core.py  ← 核心数据结构
```

## 2.2 SamplingParams - 采样参数

控制 token 生成行为的参数。

```python
@dataclass
class SamplingParams:
    top_k: int = 1           # Top-K 采样，1 表示贪婪解码
    ignore_eos: bool = False # 是否忽略 EOS token
    temperature: float = 0.0 # 温度，0 表示确定性输出
    max_tokens: int = 1024   # 最大生成 token 数
```

### 采样策略说明:
- **temperature = 0**: 贪婪解码，总是选择概率最高的 token
- **temperature > 0**: 按概率分布采样，值越大随机性越强
- **top_k**: 只从概率最高的 k 个 token 中采样

In [ ]:
from dataclasses import dataclass
from typing import List, Literal
import torch

@dataclass
class SamplingParams:
    """控制 token 生成的采样参数"""
    top_k: int = 1           # Top-K 采样
    ignore_eos: bool = False # 是否忽略 EOS token
    temperature: float = 0.0 # 采样温度
    max_tokens: int = 1024   # 最大生成长度

# 示例：不同的采样配置
greedy = SamplingParams(temperature=0.0, top_k=1)
creative = SamplingParams(temperature=0.8, top_k=50)
diverse = SamplingParams(temperature=1.2, top_k=100)

print("贪婪解码:", greedy)
print("创意模式:", creative)
print("多样性模式:", diverse)

In [ ]:
# 模拟采样过程
import torch.nn.functional as F

def sample_token(logits: torch.Tensor, params: SamplingParams) -> int:
    """根据采样参数从 logits 中采样一个 token"""
    if params.temperature == 0:
        # 贪婪解码
        return logits.argmax().item()
    
    # 应用温度
    scaled_logits = logits / params.temperature
    
    # Top-K 过滤
    if params.top_k > 0:
        values, indices = torch.topk(scaled_logits, min(params.top_k, len(scaled_logits)))
        scaled_logits = torch.full_like(scaled_logits, float('-inf'))
        scaled_logits.scatter_(0, indices, values)
    
    # 采样
    probs = F.softmax(scaled_logits, dim=-1)
    return torch.multinomial(probs, 1).item()

# 模拟 vocab_size=10 的 logits
torch.manual_seed(42)
logits = torch.randn(10)
print(f"Logits: {logits}")
print(f"Softmax 概率: {F.softmax(logits, dim=-1)}")
print()

# 不同采样策略的结果
print(f"贪婪解码结果: token_{sample_token(logits, greedy)}")
print(f"创意模式结果 (多次采样会不同):")
for i in range(5):
    print(f"  第 {i+1} 次: token_{sample_token(logits, creative)}")

## 2.3 Req - 请求类

`Req` 是系统中最重要的类之一，表示一个正在处理的推理请求。

### 关键属性:

```python
class Req:
    host_ids: torch.Tensor    # CPU 上的 token IDs (包含输入 + 已生成的 token)
    table_idx: int            # 在 Page Table 中的索引
    cached_len: int           # 已经在 KV Cache 中的 token 数量
    device_len: int           # 当前在设备上的 token 数量
    max_device_len: int       # 最大设备长度 (input_len + max_tokens)
    uid: int                  # 唯一请求 ID
    sampling_params: SamplingParams
    cache_handle: BaseCacheHandle  # Radix Cache 的句柄
```

### 关键关系:
```
0 <= cached_len < device_len <= max_device_len

extend_len = device_len - cached_len  # 需要扩展的 token 数
remain_len = max_device_len - device_len  # 剩余可生成的 token 数
```

In [ ]:
# 简化版的 Req 类实现
class Req:
    def __init__(
        self,
        *,
        input_ids: torch.Tensor,  # CPU tensor
        table_idx: int,
        cached_len: int,
        output_len: int,
        uid: int,
        sampling_params: SamplingParams,
    ):
        assert input_ids.is_cpu, "input_ids must be on CPU"
        
        self.host_ids = input_ids
        self.table_idx = table_idx
        self.cached_len = cached_len
        self.device_len = len(input_ids)
        self.max_device_len = len(input_ids) + output_len
        self.uid = uid
        self.sampling_params = sampling_params
        
        # 验证不变量
        assert 0 <= self.cached_len < self.device_len <= self.max_device_len
    
    @property
    def remain_len(self) -> int:
        """剩余可生成的 token 数"""
        return self.max_device_len - self.device_len
    
    @property
    def extend_len(self) -> int:
        """需要新计算的 token 数 (不在 cache 中的)"""
        return self.device_len - self.cached_len
    
    def complete_one(self):
        """完成一个 token 的生成"""
        self.cached_len = self.device_len
        self.device_len += 1
    
    def append_host(self, next_token: torch.Tensor):
        """将新生成的 token 追加到 host_ids"""
        self.host_ids = torch.cat([self.host_ids, next_token])
    
    def can_decode(self) -> bool:
        """是否还能继续解码"""
        return self.remain_len > 0
    
    def __repr__(self):
        return (f"Req(uid={self.uid}, table_idx={self.table_idx}, "
                f"cached_len={self.cached_len}, device_len={self.device_len}, "
                f"max_device_len={self.max_device_len})")

In [ ]:
# 创建一个示例请求
input_tokens = torch.tensor([1, 2, 3, 4, 5], dtype=torch.int32)  # 5 个输入 token
req = Req(
    input_ids=input_tokens,
    table_idx=0,
    cached_len=0,  # 没有缓存
    output_len=10,  # 最多生成 10 个 token
    uid=1,
    sampling_params=SamplingParams()
)

print("初始状态:")
print(f"  {req}")
print(f"  extend_len (需要计算): {req.extend_len}")
print(f"  remain_len (剩余可生成): {req.remain_len}")
print(f"  can_decode: {req.can_decode()}")

In [ ]:
# 模拟 Prefill 后的状态变化
print("\n=== Prefill 阶段 ===")
print("处理所有 5 个输入 token...")
# Prefill 后，所有输入 token 都被处理并缓存
req.complete_one()  # 生成第一个 output token
req.append_host(torch.tensor([100]))  # 假设生成的 token 是 100

print(f"\nPrefill 后状态:")
print(f"  {req}")
print(f"  host_ids: {req.host_ids.tolist()}")
print(f"  extend_len: {req.extend_len}")
print(f"  remain_len: {req.remain_len}")

In [ ]:
# 模拟 Decode 阶段
print("\n=== Decode 阶段 ===")
step = 1
while req.can_decode() and step <= 5:  # 只模拟 5 步
    print(f"\nStep {step}:")
    print(f"  处理位置 {req.device_len - 1} 的 token...")
    
    # 模拟生成一个新 token
    new_token = torch.tensor([100 + step])
    req.complete_one()
    req.append_host(new_token)
    
    print(f"  生成 token: {new_token.item()}")
    print(f"  当前状态: cached_len={req.cached_len}, device_len={req.device_len}")
    print(f"  剩余可生成: {req.remain_len}")
    step += 1

print(f"\n最终 host_ids: {req.host_ids.tolist()}")

## 2.4 Batch - 批处理类

`Batch` 将多个请求组合在一起进行批处理，提高 GPU 利用率。

```python
class Batch:
    reqs: List[Req]                    # 包含的请求列表
    phase: Literal["prefill", "decode"]  # 当前阶段
    input_ids: torch.Tensor            # 连接后的输入 token (由 scheduler 设置)
    out_loc: torch.Tensor              # KV cache 输出位置 (由 scheduler 设置)
    padded_reqs: List[Req]             # 填充后的请求列表 (用于 CUDA Graph)
    attn_metadata: BaseAttnMetadata    # 注意力元数据 (由 attention backend 设置)
```

In [ ]:
class Batch:
    def __init__(self, *, reqs: List[Req], phase: Literal["prefill", "decode"]):
        self.reqs = reqs
        self.phase = phase
        # 以下字段由 scheduler 设置
        self.input_ids: torch.Tensor = None
        self.out_loc: torch.Tensor = None
        self.padded_reqs: List[Req] = reqs  # 默认无填充
    
    @property
    def is_prefill(self) -> bool:
        return self.phase == "prefill"
    
    @property
    def is_decode(self) -> bool:
        return self.phase == "decode"
    
    @property
    def size(self) -> int:
        """实际请求数量"""
        return len(self.reqs)
    
    @property
    def padded_size(self) -> int:
        """填充后的请求数量 (用于 CUDA Graph)"""
        return len(self.padded_reqs)

In [ ]:
# 创建多个请求
requests = []
for i in range(3):
    req = Req(
        input_ids=torch.tensor([1, 2, 3, 4, 5 + i], dtype=torch.int32),
        table_idx=i,
        cached_len=0,
        output_len=10,
        uid=i,
        sampling_params=SamplingParams()
    )
    requests.append(req)

# 创建 Prefill Batch
prefill_batch = Batch(reqs=requests, phase="prefill")
print("=== Prefill Batch ===")
print(f"批大小: {prefill_batch.size}")
print(f"是 Prefill: {prefill_batch.is_prefill}")
print(f"是 Decode: {prefill_batch.is_decode}")
print("包含的请求:")
for req in prefill_batch.reqs:
    print(f"  {req}")

In [ ]:
# 模拟 Scheduler 设置 input_ids
# 在 Prefill 阶段，需要连接所有请求的 extend tokens
def prepare_prefill_batch(batch: Batch):
    """模拟 scheduler 准备 prefill batch"""
    all_tokens = []
    for req in batch.reqs:
        # 只取需要扩展的部分 (cached_len 到 device_len)
        extend_tokens = req.host_ids[req.cached_len:req.device_len]
        all_tokens.append(extend_tokens)
    batch.input_ids = torch.cat(all_tokens)
    return batch

prefill_batch = prepare_prefill_batch(prefill_batch)
print("\nScheduler 设置后:")
print(f"input_ids 形状: {prefill_batch.input_ids.shape}")
print(f"input_ids: {prefill_batch.input_ids.tolist()}")
print("\n每个请求贡献的 tokens:")
for i, req in enumerate(prefill_batch.reqs):
    print(f"  Req {i}: {req.extend_len} tokens")

In [ ]:
# 模拟 Decode Batch
# Decode 阶段每个请求只处理一个 token
for req in requests:
    req.cached_len = req.device_len  # 假设 prefill 完成
    req.device_len += 1  # 每个请求正在生成新 token

decode_batch = Batch(reqs=requests, phase="decode")

def prepare_decode_batch(batch: Batch):
    """模拟 scheduler 准备 decode batch"""
    # Decode 时每个请求只有 1 个 token 需要处理
    # 但这里我们需要最后一个生成的 token (需要从 GPU 读取)
    # 简化：假设每个请求用一个占位符
    batch.input_ids = torch.zeros(batch.size, dtype=torch.int32)
    return batch

decode_batch = prepare_decode_batch(decode_batch)
print("=== Decode Batch ===")
print(f"批大小: {decode_batch.size}")
print(f"input_ids 形状: {decode_batch.input_ids.shape}")
print("每个请求处理 1 个 token")

## 2.5 Context - 全局上下文

`Context` 是一个单例类，持有推理过程中的全局状态。

```python
class Context:
    page_table: torch.Tensor      # [max_req, max_seq_len] 页表
    kv_cache: BaseKVCache         # KV Cache 存储
    attn_backend: BaseAttnBackend # Attention 后端
    _batch: Batch                 # 当前正在处理的 batch
```

### 为什么使用全局单例?
- 避免在函数调用中传递大量参数
- Attention 层需要访问 KV Cache 和元数据
- 模型的每一层都需要访问相同的上下文

In [ ]:
from contextlib import contextmanager

class Context:
    def __init__(self, page_table: torch.Tensor):
        self._batch = None
        self.page_table = page_table
    
    def set_batch(self, batch: Batch):
        """设置当前 batch"""
        assert self._batch is None, "Batch already set"
        self._batch = batch
    
    def reset_batch(self):
        """重置当前 batch"""
        assert self._batch is not None, "No batch to reset"
        self._batch = None
    
    @contextmanager
    def forward_batch(self, batch: Batch):
        """上下文管理器，用于安全地设置和重置 batch"""
        self.set_batch(batch)
        try:
            yield
        finally:
            self.reset_batch()
    
    @property
    def batch(self) -> Batch:
        """获取当前 batch"""
        assert self._batch is not None, "No batch is set"
        return self._batch

# 全局单例
_GLOBAL_CTX = None

def set_global_ctx(ctx: Context):
    global _GLOBAL_CTX
    _GLOBAL_CTX = ctx

def get_global_ctx() -> Context:
    assert _GLOBAL_CTX is not None, "Global context not set"
    return _GLOBAL_CTX

In [ ]:
# 使用示例
page_table = torch.zeros((256, 4096), dtype=torch.int32)  # 256 个请求槽，每个最多 4096 tokens
ctx = Context(page_table=page_table)
set_global_ctx(ctx)

# 模拟前向传播
batch = Batch(reqs=requests[:2], phase="decode")

with ctx.forward_batch(batch):
    # 在 forward_batch 上下文中，可以安全地访问当前 batch
    current_batch = get_global_ctx().batch
    print(f"当前处理的 batch 大小: {current_batch.size}")
    print(f"当前阶段: {current_batch.phase}")

# 上下文结束后，batch 被自动重置
try:
    _ = ctx.batch
except AssertionError as e:
    print(f"\n上下文结束后访问 batch: {e}")

## 2.6 Page Table 详解

Page Table 是 Paged Attention 的核心数据结构，用于将逻辑地址映射到物理 KV Cache 位置。

```
┌─────────────────────────────────────────────────────────────┐
│                      Page Table                             │
│                 [max_req, max_seq_len]                      │
├─────┬─────┬─────┬─────┬─────┬─────┬─────────────────────────┤
│     │  0  │  1  │  2  │  3  │  4  │  ...                    │
├─────┼─────┼─────┼─────┼─────┼─────┼─────────────────────────┤
│  0  │ 10  │ 11  │ 12  │ 13  │ 14  │  Req 0 的 KV 位置       │
│  1  │ 20  │ 21  │ 22  │  0  │  0  │  Req 1 的 KV 位置       │
│  2  │ 30  │ 31  │  0  │  0  │  0  │  Req 2 的 KV 位置       │
│ ... │ ... │ ... │ ... │ ... │ ... │                         │
└─────┴─────┴─────┴─────┴─────┴─────┴─────────────────────────┘

page_table[req_idx, token_idx] = kv_cache_physical_location
```

这种设计的优势:
1. **动态分配**: 只为实际使用的 token 分配 KV Cache
2. **内存效率**: 避免预分配最大长度的连续内存
3. **灵活回收**: 请求完成后可以立即回收内存

In [ ]:
# Page Table 可视化
import numpy as np

def visualize_page_table(page_table: torch.Tensor, num_reqs: int = 4, num_tokens: int = 8):
    """可视化 Page Table 的一部分"""
    subset = page_table[:num_reqs, :num_tokens].numpy()
    
    print("Page Table 可视化:")
    print("="*50)
    print(f"{'Req':>4} |" + "".join(f"{i:>6}" for i in range(num_tokens)))
    print("-"*50)
    for i in range(num_reqs):
        values = "".join(f"{v:>6}" for v in subset[i])
        print(f"{i:>4} |{values}")

# 模拟分配 KV Cache
page_table = torch.zeros((10, 16), dtype=torch.int32)

# 为 Req 0 分配 5 个位置
page_table[0, :5] = torch.tensor([100, 101, 102, 103, 104])

# 为 Req 1 分配 3 个位置
page_table[1, :3] = torch.tensor([200, 201, 202])

# 为 Req 2 分配 7 个位置
page_table[2, :7] = torch.tensor([300, 301, 302, 303, 304, 305, 306])

visualize_page_table(page_table)

## 2.7 数据结构之间的关系

```
┌────────────────────────────────────────────────────────────────────────┐
│                          Context (全局单例)                            │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ page_table: [max_req, max_seq_len] → KV Cache 物理位置           │  │
│  │ kv_cache: 实际的 KV 数据存储                                     │  │
│  │ attn_backend: 注意力计算后端                                     │  │
│  │ _batch: 当前正在处理的 Batch                                     │  │
│  └──────────────────────────────────────────────────────────────────┘  │
│                                │                                       │
│                                ▼                                       │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │                    Batch (批处理)                                │  │
│  │  reqs: List[Req]          ──────────────────────────────┐       │  │
│  │  phase: "prefill" | "decode"                            │       │  │
│  │  input_ids: 连接后的输入 tokens                         │       │  │
│  │  out_loc: KV Cache 输出位置                             │       │  │
│  └─────────────────────────────────────────────────────────┼───────┘  │
│                                                            │          │
│                                                            ▼          │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │                    Req (单个请求)                                │  │
│  │  host_ids: CPU 上的 token IDs                                   │  │
│  │  table_idx: 在 page_table 中的索引 ─────────────────────────────│──┼──► page_table[table_idx]
│  │  cached_len / device_len / max_device_len                       │  │
│  │  sampling_params: 采样参数                                      │  │
│  │  cache_handle: Radix Cache 句柄                                 │  │
│  └──────────────────────────────────────────────────────────────────┘  │
└────────────────────────────────────────────────────────────────────────┘
```

## 2.8 重点总结

### 核心要点:

1. **SamplingParams**: 控制生成行为 (温度、top_k 等)

2. **Req** 的关键属性:
   - `cached_len`: 已缓存的 token 数
   - `device_len`: 当前设备上的 token 数
   - `extend_len = device_len - cached_len`: 需要新计算的 token 数
   - `remain_len = max_device_len - device_len`: 剩余可生成的 token 数

3. **Batch** 的两个阶段:
   - `prefill`: 处理多个 token，计算密集
   - `decode`: 每个请求只处理一个 token，内存密集

4. **Context** 作为全局单例:
   - 持有 page_table、kv_cache、attn_backend
   - 使用 `forward_batch` 上下文管理器安全地设置/重置 batch

5. **Page Table** 的作用:
   - 将逻辑位置映射到物理 KV Cache 位置
   - 支持动态分配和回收

---

**下一步**: [Module 3: 模型层](./03_model_layers.ipynb) - 学习 Linear, Embedding, RMSNorm, RoPE 等基础层的实现。